In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

In [ ]:

import yaml
config = yaml.safe_load(open('../config.yaml', 'r'))

import os
os.environ["CUDA_VISIBLE_DEVICES"] = config['gpu']
import torch
from src.model.models_dsfno_3d_noncomplex import DSFNO
from src.dataloader.dataloader_3d import dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
""""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dsfno = DSFNO(in_channel = 5, modes = 8, n_channels=16, n_residual_blocks=2, n_operator_blocks=1).to(device)
dsfno.load_state_dict(torch.load('../model/dsfno_model', weights_only=True))
"""

In [ ]:
max_samples = 30
dataset = dataset_sr(max_samples=max_samples, snapshot_index=3)

In [ ]:
hr_state, lr_state_tensor = dataset[2]
hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()


In [ ]:
dsfno_model = DSFNO(in_channel=5, 
                    modes=config['dsfno']['modes'],
                    n_channels=config['dsfno']['n_channels'],
                    n_residual_blocks=config['dsfno']['n_residual_blocks'],
                    n_operator_blocks=config['dsfno']['n_operator_blocks'], 
                    apply_constraint=config['dsfno']['apply_constraint'])


In [ ]:
from src.training.training_cnn import training_model

In [ ]:
losses, dsfno_model = training_model(
    dsfno_model, 
    None, 
    config['training']['learning_rate'], 
    config['training']['epochs'], 
    train_loader,
    use_amp=False,
    upsample_factor = 4
    )


In [ ]:
hr_state, lr_state_tensor = dataset[2]
output = dsfno_model(torch.unsqueeze(lr_state_tensor,0), 4)
print(output)

In [ ]:
n_plots = 3
random_list = random.sample(range(max_samples), n_plots)
scale_factor = 4
sr_factor = 2
fig, axes = plt.subplots(n_plots, 9, figsize=(20, n_plots * 3))

# plot cuts at this z level
z_level = 60
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor
for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
for i, list_i in enumerate(random_list):
    hr_state, lr_state_tensor = dataset[list_i]
    with torch.no_grad():
        sr_state = torch.squeeze(dsfno(torch.unsqueeze(lr_state_test_tensor, 0).to(device), sr_factor),0).cpu().detach().numpy()
    hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()

    axes[i,0].imshow(lr_state[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,0].set_ylabel(f"{list_i}")
    axes[i,1].imshow(np.sqrt(lr_state[1, :, :, z_level_reduced]**2 + lr_state[2, :, :, z_level_reduced]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,2].imshow(lr_state[4, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    
    axes[i,3].imshow(hr_state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,4].imshow(np.sqrt(hr_state[1, :, :, z_level]**2 + hr_state[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,5].imshow(hr_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

    axes[i,6].imshow(sr_state[0, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,7].imshow(np.sqrt(sr_state[1, :, :, z_level_sr]**2 + sr_state[2, :, :, z_level_sr]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,8].imshow(sr_state[4, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

    if i == 0:
        axes[i,0].set_title("Density")
        axes[i,1].set_title("Velocity")
        axes[i,2].set_title("Pressure")
        axes[i,3].set_title("Density")
        axes[i,4].set_title("Velocity")
        axes[i,5].set_title("Pressure")
        axes[i,6].set_title("Density")
        axes[i,7].set_title("Velocity")
        axes[i,8].set_title("Pressure")

    # equal aspect ratio
    axes[i,0].set_aspect('equal', 'box')
    axes[i,1].set_aspect('equal', 'box')
    axes[i,2].set_aspect('equal', 'box')

In [ ]:
index = random.sample(range(max_samples), 1)
scale_factor = 4
sr_factor = 2
fig, axes = plt.subplots(3, 3, figsize=(12,12))

# plot cuts at this z level
z_level = 60
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor
for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
hr_state, lr_state_tensor = dataset[list_i]

with torch.no_grad():
    sr_state = torch.squeeze(dsfno(torch.unsqueeze(lr_state_test_tensor, 0).to(device), sr_factor),0).cpu().detach().numpy()
hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()

axes[0,0].imshow(lr_state[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
axes[0,0].set_ylabel(f"{list_i}")
axes[0,1].imshow(np.sqrt(lr_state[1, :, :, z_level_reduced]**2 + lr_state[2, :, :, z_level_reduced]**2).T, origin = "lower", extent = [0, 1, 0, 1])
axes[0,2].imshow(lr_state[4, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

axes[1,0].imshow(hr_state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
axes[1,1].imshow(np.sqrt(hr_state[1, :, :, z_level]**2 + hr_state[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
axes[1,2].imshow(hr_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

axes[2,0].imshow(sr_state[0, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
axes[2,1].imshow(np.sqrt(sr_state[1, :, :, z_level_sr]**2 + sr_state[2, :, :, z_level_sr]**2 + sr_state[3, :, :, z_level_sr]**2 ).T, origin = "lower", extent = [0, 1, 0, 1])
axes[2,2].imshow(sr_state[4, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

axes[0,0].set_title("Density")
axes[0,1].set_title("Velocity")
axes[0,2].set_title("Pressure")